# Revenue report, 2010 — as it was handed over

A colleague wrote this notebook for finance's review of 2010. It reports three things:

1. revenue in 2010,
2. how many identified customers bought, and the average revenue per identified customer,
3. every identified customer's revenue, as a table.

Finance's brief defines each of those numbers. It is below, and in the README. The cells under **The report** are
your colleague's. They run without an error. Run them, read the numbers, and then read the brief again, one sentence
at a time.

The data is `data/raw/online_retail.parquet`: every invoice line of a UK online gift shop, 1 December 2009 to
9 December 2010. One row is one line of an invoice. `Price` is in pounds per unit.

*(The other file in `data/raw/`, `countries.csv`, is the one the lecture used. This lab does not need it.)*

## The brief

> **Sales invoices.** A sales invoice has a six-digit number. An invoice number that starts with a letter is not a
> sale: `C` marks a cancellation, and `A` an accounting adjustment (its description says *Adjust bad debt*).
>
> **Revenue** is the sum of `Quantity * Price` over the lines of sales invoices.
>
> **Identified revenue** is revenue from lines that carry a `Customer ID`. Lines with no `Customer ID` are
> *unidentified*: they are part of revenue, and part of no per-customer figure.
>
> **Average revenue per identified customer** is identified revenue divided by the number of distinct
> `Customer ID`s on those same lines.
>
> **2010** means every line whose `InvoiceDate` is in 2010.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between: count it, then census it by group

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

## The report

### 1. Revenue, 2010

In [ ]:
revenue = con.sql("""
    SELECT SUM(Quantity * Price)
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
""").fetchone()[0]
print(f"Revenue, 2010: £{revenue:,.2f}")

### 2. Identified customers, and the average revenue per identified customer

In [ ]:
customers = con.sql("""
    SELECT COUNT(DISTINCT "Customer ID")
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
""").fetchone()[0]
print(f"Identified customers, 2010: {customers:,}")
print(f"Average revenue per identified customer: £{revenue / customers:,.2f}")

### 3. Revenue per identified customer, 2010 (one row per customer)

In [ ]:
con.sql("""
    SELECT "Customer ID", ROUND(SUM(Quantity * Price), 2) AS revenue
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
      AND "Customer ID" IS NOT NULL
    GROUP BY "Customer ID"
    ORDER BY revenue DESC
""").df()

---

# Your work starts here

Work top to bottom. Paste what each inspection returns into `DIAGNOSIS.md`, part 3, **before** you change anything.

## A. Inspect the lines behind the report (run these; they are supplied)

Each one looks at the same lines the report adds up: every line dated 2010. For each, write one sentence in a
comment: what did it tell you? Then hold each report number against its sentence in the brief. Which clause of the
sentence does the report's query not do?

In [ ]:
# 1. The census of the invoice number's first character: every value, and how many lines have it.
con.sql("""
    SELECT left(Invoice, 1) AS first_character,
           COUNT(*) AS lines
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
    GROUP BY first_character
    ORDER BY lines DESC
""").df()

In [ ]:
# 2. The lines with a negative quantity, by the first character of their invoice number.
con.sql("""
    SELECT left(Invoice, 1) AS first_character,
           COUNT(*) AS lines_with_negative_quantity
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
      AND Quantity < 0
    GROUP BY first_character
    ORDER BY first_character
""").df()

In [ ]:
# 3. How many lines, and how many of them carry a customer?
con.sql("""
    SELECT COUNT(*) AS lines,
           COUNT("Customer ID") AS lines_with_a_customer
    FROM 'data/raw/online_retail.parquet'
    WHERE year(InvoiceDate) = 2010
""").df()

## B. The lines that count as revenue

Write the brief's first two sentences once, as a **view**: a saved query with a name, as in Lab 1. Call it
`sales_2010`: every line of a sales invoice dated in 2010. Every corrected number below reads `FROM sales_2010`.
Then run the census from A.1 again, **on your view**. A filter is a claim: read what it kept.

In [ ]:
# con.sql("""
#     CREATE OR REPLACE VIEW sales_2010 AS
#     SELECT *
#     FROM 'data/raw/online_retail.parquet'
#     WHERE ...        -- the brief's sentence, as a filter
# """)

In [ ]:
# The census from A.1 again, but FROM sales_2010.

## C. Revenue by month, 2010

Write this query yourself, from the empty cell: one row per month of 2010, with that month's revenue as the brief
defines it, in date order. **Before you run it, write in a comment how many rows you expect.**
(`date_trunc('month', InvoiceDate)` turns a timestamp into the first moment of its month: it was on a slide.)

In [ ]:
# How many rows do you expect? Then your query.

## D. Which countries had more than 100 invoices in 2010?

From the empty cell: each such country with its number of invoices, most first. An invoice is a sales invoice, as the
brief defines it. Then one sentence, in a comment: why can the condition *more than 100 invoices* not go in the
`WHERE`?

In [ ]:
# Your query. Then: why not in the WHERE?

## E. The check: does your revenue follow the brief, and does your month table add up?

Two identities. Neither one reuses your view's filter.

1. Every 2010 line has exactly one first character, so the census splits the report's old number into parts. Take
   away the lines the brief says are not sales — found by their first character, not by your view — and what is left
   must equal the revenue in your view.
2. Revenue is additive, so the twelve months from section C must add up to your total, to the penny.

Print the numbers on each side and the difference, rounded to the penny. "To the penny" means a difference under half
a penny, never `==`: two correct sums of the same lines can differ in the tenth decimal place. A difference of a penny
or more means your view or your month table kept or lost lines the brief did not ask for.

In [ ]:
# Identity 1: all 2010 lines, the C and A lines, your view. Identity 2: the months against your view.

## Now the note

Write `DIAGNOSIS.md` now, all five parts, short — before section F. Part 3 is what section A showed you; part 5 is
section E's output, pasted. The note is part of the lab; section F is not.

## F. After the note, if there is time: average revenue per identified customer

*This is Homework 1's question 7 as well. If the lab ends before you get here, you will write it there.*

The brief's sentence has a top and a bottom. Compute both from **the same lines** — the identified lines of your
view — and divide. Print the unidentified revenue beside it, with its share of revenue, so a reader knows what the
average leaves out. Then check the ratio against the per-customer table: one row per identified customer, from your
view. Its row count must equal your bottom, and its average must equal your ratio.

In [ ]:
# Top, bottom, the ratio, the unidentified share. Then the per-customer table's count and average.